# 09 - Evaluate DNABERT2 across repeated lineage-aware splits

A single lineage-held-out split is informative but unstable. This notebook reuses the frozen full-cohort DNABERT2 embeddings, regenerates grouped train/dev/test partitions across ten seeds, selects a classification threshold on each development partition, and reports test distributions.

## 1. Configuration

No transformer inference is needed. The embedding matrix is already available.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EMBEDDINGS_PATH = PROJECT_ROOT / "data" / "processed" / "dnabert2_full_embeddings.npz"
MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_public_assemblies.csv"
OUTPUT_PATH = PROJECT_ROOT / "reports" / "dnabert2_repeated_grouped_evaluation.json"
SEEDS = "42,43,44,45,46,47,48,49,50,51"
RUN_REPEATED_EVALUATION = False

print("Project root:", PROJECT_ROOT)
print("Embeddings exist:", EMBEDDINGS_PATH.exists())

## 2. Run repeated grouped evaluation

Change `RUN_REPEATED_EVALUATION` to `True`. Thresholds are selected using development isolates only; test isolates remain held out.

In [ ]:
evaluation_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "evaluate_embedding_repeated_grouped_splits.py"),
    "--embeddings",
    str(EMBEDDINGS_PATH),
    "--manifest",
    str(MANIFEST_PATH),
    "--seeds",
    SEEDS,
    "--output",
    str(OUTPUT_PATH),
]
print(" ".join(evaluation_command))
if RUN_REPEATED_EVALUATION:
    subprocess.run(evaluation_command, cwd=PROJECT_ROOT, check=True)
if OUTPUT_PATH.exists():
    report = json.loads(OUTPUT_PATH.read_text())
    print(json.dumps(report["summary"], indent=2))
    evaluations = pd.DataFrame(report["evaluations"])
    display(evaluations)

## 3. Next step

Compare repeated DNABERT2 performance with repeated grouped k-mer evaluation and a known-QAC-gene rule baseline. Only then decide whether to increase windows per isolate or add a more complex pooling model.